# 01 · Data Cleaning & Feature Engineering

Loads the raw marketing dataset (56,000 customers), profiles it for quality issues, cleans it, and engineers the derived features used throughout the project. Output: `data/marketing_clean.csv`.

In [1]:
import pandas as pd
import numpy as np
import os


In [2]:
df = pd.read_csv("data/marketing_data.csv")

In [3]:
df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,Complain,Country
0,342199,1985,Graduation,Together,59011.7,1,0,2012-11-17,3,0,...,3,4,0,0,0,0,0,0,0,Spain
1,8075450,1975,Master,Single,1730.0,1,1,2013-04-10,96,0,...,2,3,0,0,0,0,0,0,0,Spain
2,13664263,1978,Graduation,Married,98584.6,0,0,2014-01-11,99,920,...,6,3,0,0,0,0,0,0,0,Australia
3,16164787,1976,Graduation,Married,74031.5,1,0,2014-06-18,47,265,...,11,4,0,0,0,0,0,0,0,Spain
4,15815139,1981,Graduation,Divorced,52784.2,1,1,2014-05-20,0,30,...,3,6,0,0,0,0,0,0,0,Canada


In [4]:
df.shape

(56000, 28)

In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56000 entries, 0 to 55999
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   56000 non-null  int64  
 1   Year_Birth           56000 non-null  int64  
 2   Education            56000 non-null  object 
 3   Marital_Status       56000 non-null  object 
 4   Income               56000 non-null  float64
 5   Kidhome              56000 non-null  int64  
 6   Teenhome             56000 non-null  int64  
 7   Dt_Customer          56000 non-null  object 
 8   Recency              56000 non-null  int64  
 9   MntWines             56000 non-null  int64  
 10  MntFruits            56000 non-null  int64  
 11  MntMeatProducts      56000 non-null  int64  
 12  MntFishProducts      56000 non-null  int64  
 13  MntSweetProducts     56000 non-null  int64  
 14  MntGoldProds         56000 non-null  int64  
 15  NumDealsPurchases    56000 non-null 

## 1. Data Quality Checks

Check for missing values and duplicate rows before changing anything. This dataset turns out to have **no missing values and no duplicates**, so no imputation or de-duplication is needed.

In [6]:
df.isnull().sum()

ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 0
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Response               0
Complain               0
Country                0
dtype: int64

In [7]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df['ID'].duplicated().sum())

Duplicate rows: 0
Duplicate customer IDs: 0


## 2. Cleaning Categorical Values

Inspect the text columns with `value_counts()`. `Marital_Status` contains junk entries — **Alone, Absurd, YOLO**. We merge `Alone → Single` (same meaning) and relabel the nonsense values as **Unknown**, keeping the rows so their spending data is preserved. `Education` and `Country` are clean.

In [8]:
df['Marital_Status'].value_counts()

Marital_Status
Together    17703
Married     13686
Single      12238
Divorced     6459
Widow        3540
Alone         911
Absurd        733
YOLO          730
Name: count, dtype: int64

In [9]:
df['Education'].value_counts()

Education
Graduation    22741
PhD           12581
Master        10530
2n Cycle       5966
Basic          4182
Name: count, dtype: int64

In [10]:
df['Country'].value_counts()

Country
Spain           16703
Canada          10746
Saudi Arabia     8422
Australia        5677
India            4814
Germany          4432
USA              4331
Mexico            875
Name: count, dtype: int64

In [11]:
df['Marital_Status']= df['Marital_Status'].replace({
    'Alone' : 'Single',
   'Absurd' : 'Unknown',
   'YOLO' : 'Unknown'})


In [12]:
df['Marital_Status'].value_counts()

Marital_Status
Together    17703
Married     13686
Single      13149
Divorced     6459
Widow        3540
Unknown      1463
Name: count, dtype: int64

## 3. Fixing Data Types

`Dt_Customer` is stored as text. Convert it to a real `datetime` so we can compute customer tenure later.

In [13]:
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'])

In [14]:
df['Dt_Customer'].dtype

dtype('<M8[ns]')

## 4. Outlier Check

Use `describe()` to inspect numeric ranges. Ages (30–90) and income are all realistic — no outlier treatment needed.

In [15]:
df.describe()

,ID,Year_Birth,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,...,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,Complain
count,5.600000e+04,56000.000000,56000.000000,56000.000000,56000.000000,56000,56000.000000,56000.000000,56000.000000,56000.000000,...,56000.000000,56000.000000,56000.000000,56000.000000,56000.000000,56000.000000,56000.000000,56000.000000,56000.000000,56000.000000
mean,8.389352e+06,1971.666696,57252.189521,0.539911,0.362143,2013-08-15 07:30:18.514285568,63.221107,246.981482,16.152661,268.294018,...,2.110750,4.706411,5.170107,0.062393,0.056821,0.045661,0.134446,0.014411,0.147589,0.007625
min,3.600000e+01,1936.000000,1730.000000,0.000000,0.000000,2012-07-30 00:00:00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.187372e+06,1963.000000,28252.025000,0.000000,0.000000,2013-01-24 00:00:00,35.000000,0.000000,0.000000,42.000000,...,0.000000,3.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,8.383784e+06,1973.000000,58838.550000,1.000000,0.000000,2013-08-19 00:00:00,71.000000,64.000000,0.000000,107.000000,...,1.000000,4.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.258576e+07,1981.000000,86930.650000,1.000000,1.000000,2014-03-10 00:00:00,93.000000,353.000000,15.000000,363.000000,...,3.000000,6.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.677716e+07,1996.000000,258027.500000,2.000000,2.000000,2014-06-29 00:00:00,99.000000,1493.000000,199.000000,1341.000000,...,14.000000,13.000000,10.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,4.844638e+06,12.211066,34307.247999,0.521349,0.510567,NaN,31.157066,358.569481,33.931463,342.165712,...,2.357824,2.896366,2.555911,0.241870,0.231503,0.208750,0.341134,0.119178,0.354696,0.086988


## 5. Feature Engineering

Create the derived features the analysis needs: **Age, Total_Spend, Total_Purchase, Children, Customer_Tenure**, the creative features **Avg_Order_Value** and **Web_Conversion_Rate**, and binned **Age_Band / Income_Band**.

In [16]:
df['Age'] = 2026 - df['Year_Birth']

In [17]:
df['Age'].describe()


count    56000.000000
mean        54.333304
std         12.211066
min         30.000000
25%         45.000000
50%         53.000000
75%         63.000000
max         90.000000
Name: Age, dtype: float64

In [18]:
mnt_cols = ['MntWines', 'MntFruits', 'MntMeatProducts',
            'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']

df['Total_Spend'] = df[mnt_cols].sum(axis=1)

In [19]:
df[mnt_cols + ['Total_Spend']].head()

,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,Total_Spend
0,0,2,45,6,0,16,69
1,0,0,19,1,19,0,39
2,920,0,339,0,115,138,1512
3,265,0,130,7,1,75,478
4,30,3,55,242,0,0,330


In [20]:
purchase_cols= ['NumDealsPurchases','NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']

df['Total_Purchase'] = df[purchase_cols].sum(axis=1)

In [21]:
df[purchase_cols +['Total_Purchase']].head()

,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,Total_Purchase
0,3,2,5,3,13
1,1,1,0,2,4
2,3,2,1,6,12
3,1,4,0,11,16
4,1,3,2,3,9


In [22]:
df['Children'] = df['Kidhome'] +df['Teenhome']

In [23]:
df[['Kidhome','Teenhome','Children']].head()

,Kidhome,Teenhome,Children
0,1,0,1
1,1,1,2
2,0,0,0
3,1,0,1
4,1,1,2


In [24]:
reference_date = pd.Timestamp('2026-07-01')
df['Customer_Tenure'] = (reference_date - df['Dt_Customer']).dt.days

In [25]:
df[['Dt_Customer', 'Customer_Tenure']].head()

,Dt_Customer,Customer_Tenure
0,2012-11-17,4974
1,2013-04-10,4830
2,2014-01-11,4554
3,2014-06-18,4396
4,2014-05-20,4425


In [26]:
df['Avg_Order_Value'] = (df['Total_Spend'] / df['Total_Purchase'].replace(0, np.nan)).round(2)

In [27]:
df[['Total_Spend', 'Total_Purchase', 'Avg_Order_Value']].head()

,Total_Spend,Total_Purchase,Avg_Order_Value
0,69,13,5.31
1,39,4,9.75
2,1512,12,126.00
3,478,16,29.88
4,330,9,36.67


In [28]:
df['Web_Conversion_Rate']= (df['NumWebPurchases']/ df['NumWebVisitsMonth'].replace(0,np.nan)).round(2)

In [29]:
df[['NumWebPurchases', 'NumWebVisitsMonth', 'Web_Conversion_Rate']].head()

,NumWebPurchases,NumWebVisitsMonth,Web_Conversion_Rate
0,2,4,0.50
1,1,3,0.33
2,2,3,0.67
3,4,4,1.00
4,3,6,0.50


In [30]:
df['Age_Band'] = pd.cut( df['Age'],bins = [0, 29, 39, 49, 59, 69, 200],labels = ['<30', '30-39', '40-49', '50-59', '60-69', '70+'])

In [31]:
df['Age_Band'].value_counts()

Age_Band
50-59    16208
40-49    15611
60-69    11179
70+       7164
30-39     5838
<30          0
Name: count, dtype: int64

In [32]:
df['Income_Band']= pd.cut( df['Income'], bins= [0,25000,50000,75000,100000,float('inf')], labels=['<25k', '25-50k', '50-75k', '75-100k', '100k+'])



In [33]:
df['Income_Band'].value_counts()

Income_Band
<25k       12677
50-75k     12575
75-100k    12505
25-50k     10966
100k+       7277
Name: count, dtype: int64

In [34]:
df.columns


Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Response', 'Complain', 'Country', 'Age', 'Total_Spend',
       'Total_Purchase', 'Children', 'Customer_Tenure', 'Avg_Order_Value',
       'Web_Conversion_Rate', 'Age_Band', 'Income_Band'],
      dtype='object')

## 6. Export Cleaned Data

Save the cleaned, feature-engineered dataset to `data/marketing_clean.csv` for the EDA, segmentation, and SQL stages.

In [35]:
df.to_csv('data/marketing_clean.csv', index=False)
print("Saved! Shape:", df.shape)

Saved! Shape: (56000, 37)


In [36]:
print(os.path.exists('data/marketing_clean.csv'))

True
